# Calculations and demos of the perturbation theory

- Bernardeau et al (https://arxiv.org/pdf/astro-ph/0112551)
- MUSIC validation
- MUSIC-like whitenoise generation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Cosmology

## Fourier grid 

In [ ]:
def cubic_voxels(nmesh, Lbox, periodic, silent=False):
    '''
    Defines a rectangular cuboid mesh with the specified number of
    voxels in each dimensions, ensuring that the voxels are cubic.
    The function calculates the number of voxels in each dimension
    (Nx, Ny, Nz) based on the shortest dimension of the cuboid and
    scales the other dimensions accordingly.

    Parameters:
    -----------
    nmesh : int
        Number of voxels in the shortest dimension.
    Lbox : float or list of float
        Length of the box in each dimension [Lx, Ly, Lz].
    periodic : bool or list of bool
        If True, the box is periodic in all dimensions. Arbitrary
        combinations of periodic and open boundary conditions can be
        specified for each dimension as a list of three booleans.
    silent : bool
        If True, suppresses output messages.
    '''
    if not isinstance(Lbox, (list, tuple)):
        Lbox = (Lbox,) * 3
    if not isinstance(periodic, (list, tuple)):
        periodic = (periodic,) * 3
    ref_L = np.min(Lbox)
    mesh = np.ceil(Lbox / (ref_L / nmesh)).astype(int)
    mesh = (mesh + mesh % 2).astype(int)  # Ensure even number of voxels
    mesh += 1 - np.array(periodic, dtype=int)  # Add 1 if open boundary condition
    if not silent:
        print('Mesh: Nx={}, Ny={}, Nz={}'.format(*mesh))
    dk = ref_L / mesh[Lbox.index(ref_L)]
    if not silent:
        print(f'Step size: {dk}')
    return dk, mesh

In [ ]:
def fourier_grid(nmesh, Lbox, periodic, hermitian=False, silent=True):
    '''TODO'''
    dk, nvox = cubic_voxels(nmesh, Lbox, periodic, silent=silent)
    
    kx = np.fft.fftfreq(nvox[0]) * 2*np.pi / dk
    ky = np.fft.fftfreq(nvox[1]) * 2*np.pi / dk
    
    if hermitian:
        kz = np.fft.rfftfreq(nvox[2]) * 2*np.pi / dk
    else:
        kz = np.fft.fftfreq(nvox[2]) * 2*np.pi / dk
    kvec = np.array(np.meshgrid(kx, ky, kz, indexing='ij'))
    kmod = np.linalg.norm(kvec, axis=0)
    return kvec, kmod

## 1LPT - Zel'dovich approximation

In [ ]:
def zeldovich(x, Lbox, density_field, dD1, h, counter=False):
    '''TODO'''
    nmesh = np.min(density_field.shape)
    overdensity_field = compute_overdensity(density_field)
    kvec, kmod = fourier_grid(nmesh, Lbox, hermitian=True)
    delta_k = np.fft.rfftn(overdensity_field)
    delta_k = delta_k * np.exp(1j * np.pi) if counter else delta_k
    mask = kmod > 0.0  # Avoid division by zero at k=0
    xpert = np.zeros_like(x, dtype=np.float32)
    v = np.zeros_like(x, dtype=np.float32)

    for i, xi in enumerate(('x', 'y', 'z')):
        psi1_ki = np.zeros_like(kmod, dtype=complex)
        psi1_ki[mask] = -1j * kvec[i, mask] / (kmod[mask] ** 2) * delta_k[mask]
        disp_field = np.fft.irfftn(psi1_ki, s=density_field.shape)
        max_disp = np.max(np.abs(disp_field))
        print(f"Maximal '{xi}' displacement: {max_disp*1000:.3f} kpc/h; "
              f"in units of mean particle separation: {max_disp * nmesh / Lbox:.3f}")
        disp_field_interp = interpolate_field(x, Lbox, disp_field)
        xpert[:, i] = x[:, i] + disp_field_interp
        v[:, i] = disp_field_interp * dD1
    return np.mod(xpert, Lbox), v / h

## 2LPT

In [ ]:
omega_m = 0.3
omega_m**(-1/143)